In [ ]:
!pip install torch_geometric
!pip install nibabel
!pip install nilearn
!pip install scipy
!pip install pingouin

In [ ]:

import torch
from torch_geometric.nn import GCNConv, global_mean_pool
import torch.nn.functional as F
import numpy as np
import pandas as pd
import sys
from torch.utils.data import Dataset
from torch_geometric.loader import DataLoader
import torch.nn as nn
import torch.optim as optim
import glob
import os
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import time
from torch_geometric.nn import NNConv as MyNNConv, TopKPooling
import scipy.io
import scipy.sparse as sp
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from FVE_GCN_utils import *
from FVE_GCN_script import *

import nibabel as nib
%load_ext autoreload
%autoreload 2


In [ ]:
python FVE_GCN_script.py --job_nickname "FVE_base_partial" --model_type "base" \
  --lr 0.005 --weight_decay 0.0005 --epochs 50 --patience 999  \
  --partial_dat --norm_y --scaler_name "minmax" --batch_size 20

# FreeSurfer Mapping

In [ ]:
import numpy as np
import scipy.io
from nilearn import datasets, surface
from scipy.spatial import distance
from scipy.optimize import linear_sum_assignment

partial_dat = False
partial_tsa_dat = False
icld_age_sex = True


if partial_dat:
    FVE_df_org = pd.read_csv("data/FVE_dat_partial.csv")
    SurfeView_surfaces = scipy.io.loadmat("data/SurfeView_surfaces.mat")
    non_surface_area_vars = ["nihtbx_cryst_uncorrected"]
    dat_type = "GCN_partial"
    dat_type2 = "partial"
elif partial_tsa_dat:
    FVE_df_org = pd.read_csv("data/FVE_dat_partial_tsa.csv")
    SurfeView_surfaces = scipy.io.loadmat("data/SurfeView_surfaces.mat")
    non_surface_area_vars = ["nihtbx_cryst_uncorrected"]
    dat_type = "GCN_partial_tsa"
    dat_type2 = "partial_tsa"
else:
    FVE_df_org = pd.read_csv("data/FVE_dat.csv")
    SurfeView_surfaces = scipy.io.loadmat("data/SurfeView_surfaces.mat")
    non_surface_area_vars = ["interview_age", "sex_2", "nihtbx_cryst_uncorrected"]
    dat_type2 = "regular"
    if icld_age_sex:
        dat_type = "GCN_cov"
    else:
        dat_type = "GCN"

# original mapping
SurfeView_surfaces = scipy.io.loadmat("data/SurfeView_surfaces.mat")
vertices_coords_orgs, faces_orgs = load_surface_mesh(SurfeView_surfaces)
vertices_coords_orgs_left = vertices_coords_orgs[:10242]
vertices_coords_orgs_right = vertices_coords_orgs[10242:]
print(vertices_coords_orgs.shape, faces_orgs.shape)

# load FreeSurfer 20K resolution
fsaverage = datasets.fetch_surf_fsaverage('fsaverage5')

coords_fs_left, faces_fs_left = surface.load_surf_mesh(fsaverage.pial_left)
coords_fs_right, faces_fs_right = surface.load_surf_mesh(fsaverage.pial_right)



vertices_coords_fs = np.vstack([coords_fs_left, coords_fs_right])
faces_fs = np.vstack([faces_fs_left, faces_fs_right+10242])

print(coords_fs_left.shape, coords_fs_right.shape)
print(vertices_coords_fs.shape, faces_fs.shape)


FVE_df_old_X = FVE_df_org.dropna()

left_cols = [c for c in FVE_df_old_X.columns if c.endswith('_l')]
right_cols = [c for c in FVE_df_old_X.columns if c.endswith('_r')]
FVE_df_fs = map_to_fs(FVE_df_old_X, coords_fs_left, coords_fs_right, vertices_coords_orgs_left, vertices_coords_orgs_right, non_surface_area_vars)

FVE_df_fs

In [ ]:
from nilearn import plotting, surface

subject_idx = 0
left_data = FVE_df_fs.loc[subject_idx, left_cols].values
right_data = FVE_df_fs.loc[subject_idx, right_cols].values

# Load fsaverage5 pial mesh
fsaverage = datasets.fetch_surf_fsaverage('fsaverage5')
pial_left = fsaverage.pial_left
pial_right = fsaverage.pial_right
infl_left = fsaverage.infl_left
infl_right = fsaverage.infl_right

# Plot left hemisphere
plotting.plot_surf_stat_map(
    surf_mesh=pial_left,
    stat_map=left_data,
    hemi='left',
    colorbar=True,
    title='Left Hemisphere Surface Area',
    cmap='viridis'
)

# Plot right hemisphere
plotting.plot_surf_stat_map(
    surf_mesh=pial_right,
    stat_map=right_data,
    hemi='right',
    colorbar=True,
    title='Right Hemisphere Surface Area',
    cmap='viridis'
)

plotting.show()

In [ ]:
import nibabel as nib

# Load the CIFTI-2 dense label file
img = nib.load("data_out/abcd_template_matching_combined_clusters_thresh0.61.dlabel.nii")

# Get the data (shape = (1, 91282))
data = img.get_fdata().astype(int).flatten()


# Get the label axis (axis 0)
label_axis = img.header.get_axis(0)
print(f"Label axis type: {type(label_axis)}")

# Handle nibabel version differences
label_table = label_axis.label[0]

print(f"Number of labels: {len(label_table)}")
print(f"Total vertices: {len(data)}")

# Each label entry is a tuple: (name, (R,G,B,A))
for key, value in label_table.items():
    label_name, rgba = value
    print(f"Label {key}: '{label_name}', RGBA={rgba}")


# Dictionary: ROI name -> vertex indices
roi_vertices = {}

for label_value, (label_name, rgba) in label_table.items():
    # find vertices belonging to this ROI
    indices = np.where(data == label_value)[0]
    roi_vertices[label_name] = indices
    print(f"{label_name}: {len(indices)} vertices")


roi_vertices

In [ ]:
from nilearn import plotting, surface
label_L = nib.load("data_out/abcd_labels.L.10k.label.gii")
label_R = nib.load("data_out/abcd_labels.R.10k.label.gii")


data_L = label_L.darrays[0].data
data_R = label_R.darrays[0].data

print(f"Left hemisphere vertices: {len(data_L)}")
print(f"Right hemisphere vertices: {len(data_R)}")

print(f"Unique labels (L): {np.unique(data_L)}")
print(f"Unique labels (R): {np.unique(data_R)}")


fsavg_l = surface.load_surf_mesh("data_out/fs_LR.10k.L.pial.surf.gii")
fsavg_r = surface.load_surf_mesh("data_out/fs_LR.10k.R.pial.surf.gii")

brain_mapping_mean_df = summarize_by_brain_region(df=FVE_df_fs, data_l=data_L, data_r=data_R)

if dat_type2 == "regular":
    brain_mapping_mean_df = pd.concat([brain_mapping_mean_df, FVE_df_org[['interview_age', 'sex_2', 'nihtbx_cryst_uncorrected']]], axis=1)
else:
    brain_mapping_mean_df = pd.concat([brain_mapping_mean_df, FVE_df_org[['nihtbx_cryst_uncorrected']]], axis=1) 


brain_mapping_mean_df.to_csv(f"data_out/brain_mapping_mean_{dat_type2}.csv")


from sklearn.model_selection import train_test_split
train_valid, test = train_test_split(brain_mapping_mean_df, test_size=0.2, random_state=42)

train_valid.to_csv(f"data_out/brain_mapping_mean_{dat_type2}_train.csv")
test.to_csv(f"data_out/brain_mapping_mean_{dat_type2}_test.csv")


In [ ]:
fsaverage

In [ ]:
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.cm as cm

labels_L = np.unique(data_L[data_L != 0])
labels_R = np.unique(data_R[data_R != 0])
all_labels = np.union1d(labels_L, labels_R)
n_labels = len(all_labels)


colors = ['lightgrey']  # label-0 color
colors += [cm.hsv(i / n_labels) for i in range(n_labels)]

custom_cmap = mcolors.ListedColormap(colors)


In [ ]:
custom_cmap

In [ ]:
plotting.view_surf(fsavg_l, data_L, cmap=custom_cmap, colorbar=False, title="L")



In [ ]:
plotting.view_surf(fsavg_r, data_R, cmap=custom_cmap, colorbar=False, title="R")


In [ ]:
from nilearn import plotting
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, subplot_kw={'projection': '3d'}, figsize=(6, 5))
view = 'anterior'
plotting.plot_surf_roi(
    fsavg_l, roi_map=data_L,
    hemi='left', view=view,
    cmap='tab20', axes=axes[1], title="L"
)

plotting.plot_surf_roi(
    fsavg_r, roi_map=data_R,
    hemi='right', view=view,
    cmap='tab20', axes=axes[0], title="R"
)

plt.show()


plotting.plot_surf_roi(
    fsavg_l, roi_map=data_L,
    hemi='left', view=view,
    cmap='tab20', title='anterior'
)
plt.show()

plotting.plot_surf_roi(
    fsavg_r, roi_map=data_R,
    hemi='right', view=view,
    cmap='tab20', title=""
)

plt.show()


## Partial correlation

In [ ]:
import pingouin as pg
pairwise_partial_corr = train_valid.pcorr()
print("\nPairwise partial correlation matrix:\n", pairwise_partial_corr)

## brain mapping archive


In [ ]:
n_vertices = 20482  # 0 to 20481
vertex_labels = np.zeros(n_vertices, dtype=int)  # 0 = unlabeled

# Map ROI names to integers starting from 1
label_to_int = {name: idx for idx, name in enumerate(roi_vertices.keys(), start=1)}

for label_name, indices in roi_vertices.items():
    # Find the intersection with your 0-20481 vertices
    valid_indices = indices[indices < n_vertices]  # only keep indices within 0-20481
    vertex_labels[valid_indices] = label_to_int[label_name]

print(vertex_labels[:20])  # sanity check

In [ ]:
import matplotlib.pyplot as plt
plt.plot(vertex_labels)

In [ ]:
n_vertices = vertices.shape[0]
edge_set = set()

for tri in faces:
    # add edges for each triangle
    for i in range(3):
        a, b = tri[i], tri[(i+1)%3]
        edge_set.add((a,b))
        edge_set.add((b,a))  # make it undirected

# convert to torch tensor
edge_index = torch.tensor(list(edge_set), dtype=torch.long).t().contiguous()
print(edge_index.shape)  # should be (2, num_edges)

In [ ]:
for i in range(node_feat.shape[0]):        
    data = Data(
        x=node_feat[i].unsqueeze(0),  # if each vertex has only 1 feature vector
        edge_index=edge_index,
        pos=torch.tensor(vertices, dtype=torch.float32),
        y=torch.tensor(y[i], dtype=torch.float32)
    )
    graphs.append(data)

# Mapping data GCN

In [ ]:
def compute_partial_correlation(df):
    X = df.values
    n_samples, n_vars = X.shape
    
    X_std = (X - X.mean(axis=0)) / X.std(axis=0)
    
    partial_corr = np.eye(n_vars)
    
    for i in range(n_vars):
        for j in range(i + 1, n_vars):
            other_vars = [k for k in range(n_vars) if k != i and k != j]
            
            if len(other_vars) == 0:
                partial_corr[i, j] = partial_corr[j, i] = np.corrcoef(X_std[:, i], X_std[:, j])[0, 1]
            else:
                X_other = X_std[:, other_vars]
                
                XtX = X_other.T @ X_other
                XtX += np.eye(len(other_vars)) * 1e-6
                
                beta_i = np.linalg.solve(XtX, X_other.T @ X_std[:, i])
                residual_i = X_std[:, i] - X_other @ beta_i
                
                beta_j = np.linalg.solve(XtX, X_other.T @ X_std[:, j])
                residual_j = X_std[:, j] - X_other @ beta_j
                
                partial_corr[i, j] = partial_corr[j, i] = np.corrcoef(residual_i, residual_j)[0, 1]
    
    return partial_corr


def corr_to_edge_index(partial_corr, threshold=0.1):
    n_nodes = partial_corr.shape[0]
    
    edges = []
    edge_weights = []
    
    for i in range(n_nodes):
        for j in range(i + 1, n_nodes):
            corr_val = abs(partial_corr[i, j])
            if corr_val > threshold:
                edges.append([i, j])
                edges.append([j, i])
                edge_weights.append(corr_val)
                edge_weights.append(corr_val)
    
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_weights, dtype=torch.float).unsqueeze(1)
    
    return edge_index, edge_attr

In [ ]:
partial_dat = True
partial_tsa_dat = False

if partial_dat:
    bm_test = pd.read_csv("data_out/brain_mapping_mean_partial_test.csv")
    bm_train = pd.read_csv("data_out/brain_mapping_mean_partial_train.csv")
    non_surface_area_vars = ['nihtbx_cryst_uncorrected']
elif partial_tsa_dat:
    bm_test = pd.read_csv("data_out/brain_mapping_mean_partial_tsa_test.csv")
    bm_train = pd.read_csv("data_out/brain_mapping_mean_partial_tsa_train.csv")
    non_surface_area_vars = ['nihtbx_cryst_uncorrected']
else:
    bm_test = pd.read_csv("data_out/brain_mapping_mean_regular_tsa_test.csv")
    bm_train = pd.read_csv("data_out/brain_mapping_mean_regular_tsa_train.csv")
    non_surface_area_vars = ['interview_age', 'sex_2',  'nihtbx_cryst_uncorrected'] #when FVE_Dat

FVE_df_org = pd.concat([bm_train, bm_test])


In [ ]:
FVE_df_org

In [ ]:
corr_train = compute_partial_correlation(bm_train)

In [ ]:
import seaborn as sns
sns.heatmap(corr_train)

In [ ]:
edge_index, edge_attr = corr_to_edge_index(corr_train, threshold=0.1)

In [ ]:
train_graph = create_graph_data(X_all=X_train_scaled, y=torch.Tensor(y_train_scaled), partial_dat=partial_dat, partial_tsa_dat= partial_tsa_dat,
                        edge_index = edge_index, vertices = torch.FloatTensor(vertices_coords), icld_age_sex=icld_age_sex)

# Mapping the coefficients back to the original vertices

In [ ]:
!pip install rpy2

In [ ]:
import numpy as np
import pandas as pd
import scipy.io
import pickle
from pathlib import Path
#import rpy2.robjects as robjects
#from rpy2.robjects import pandas2ri

#pandas2ri.activate()

# Configuration
wd = Path(".")
output_dir = wd / "LR_output"
B = 50

# Load the vertex-to-label mapping
import nibabel as nib

# Load FreeSurfer labels
label_L = nib.load(wd / "data_out/abcd_labels.L.10k.label.gii")
label_R = nib.load(wd / "data_out/abcd_labels.R.10k.label.gii")

data_L = label_L.darrays[0].data  # 10242 vertices
data_R = label_R.darrays[0].data  # 10242 vertices

print(f"Left hemisphere vertices: {len(data_L)}")
print(f"Right hemisphere vertices: {len(data_R)}")
print(f"Unique labels (L): {np.unique(data_L)}")
print(f"Unique labels (R): {np.unique(data_R)}")

# Load original coordinates
SurfeView_surfaces = scipy.io.loadmat(wd / "data/SurfeView_surfaces.mat")

coords_lh = SurfeView_surfaces['surf_lh_pial'][0,0]['vertices'][:10242]
coords_rh = SurfeView_surfaces['surf_rh_pial'][0,0]['vertices'][:10242]

# Load FreeSurfer coordinates
from nilearn import datasets, surface

fsaverage = datasets.fetch_surf_fsaverage('fsaverage5')
coords_fs_left, _ = surface.load_surf_mesh(fsaverage.pial_left)
coords_fs_right, _ = surface.load_surf_mesh(fsaverage.pial_right)


In [ ]:
coords_lh

Mapping back to the original ordering

In [ ]:




# Load R models and extract coefficients
robjects.r(f'''
library(readr)
wd <- "{str(wd)}"
load(paste0(wd, "/PCA_output/mapping_result_all_models_{B}.Rdata"))
''')

# Get the list of models
all_models = robjects.r('all_models')

# Model types
model_types = ['mapping_train', 'mapping_p_train', 'mapping_p_tsa_train']
model_names = ['regular', 'partial', 'partial_tsa']

# Process each model type
for model_name, model_prefix in zip(model_names, model_types):
    print(f"\n{'='*60}")
    print(f"Processing {model_name} models")
    print(f"{'='*60}")
    
    # Storage for coefficients across bootstraps
    # Each bootstrap will have coefficients for each region
    all_bootstrap_coefs = []
    
    for b in range(1, B + 1):
        model_key = f'{model_prefix}{b}'
        
        if model_key not in all_models.names:
            print(f"Warning: {model_key} not found")
            continue
        
        # Extract model
        model = all_models.rx2(model_key)
        
        # Get coefficients (excluding intercept)
        coefs = robjects.r(f'coef(all_models[["{model_key}"]])')
        coef_names = coefs.names
        coef_values = np.array(coefs)
        
        # Create dictionary of region -> coefficient
        region_coefs = {}
        for name, value in zip(coef_names, coef_values):
            if name != '(Intercept)' and not name.startswith('interview_age') and not name.startswith('sex'):
                # Extract region name (format: "region_name_l" or "region_name_r")
                region_coefs[name] = value
        
        all_bootstrap_coefs.append(region_coefs)
        
        if b == 1:
            print(f"Sample regions: {list(region_coefs.keys())[:5]}")
            print(f"Total regions: {len(region_coefs)}")
    
    # Calculate mean coefficients across bootstraps
    # Get all unique region names
    all_regions = set()
    for coef_dict in all_bootstrap_coefs:
        all_regions.update(coef_dict.keys())
    
    mean_region_coefs = {}
    for region in all_regions:
        values = [coef_dict.get(region, 0) for coef_dict in all_bootstrap_coefs]
        mean_region_coefs[region] = np.mean(values)
    
    print(f"Mean coefficients calculated for {len(mean_region_coefs)} regions")
    
    # Map region coefficients to vertices (FreeSurfer space)
    vertex_coefs_fs_left = np.zeros(10242)
    vertex_coefs_fs_right = np.zeros(10242)
    
    # Get unique labels
    unique_labels_left = np.unique(data_L)
    unique_labels_right = np.unique(data_R)
    
    # Map coefficients to FreeSurfer vertices
    for label_value in unique_labels_left:
        if label_value == 0:  # Skip background/unlabeled
            continue
        
        # Find region name in coefficient dictionary
        # Need to match region names from the model to label values
        # This depends on how regions were named in brain_mapping_mean_df
        
        vertices_in_region = np.where(data_L == label_value)[0]
        
        # Try to find matching coefficient
        # Region names might be like "region_1_l", "region_2_l", etc.
        region_name = f"region_{int(label_value)}_l"  # Adjust based on actual naming
        
        if region_name in mean_region_coefs:
            vertex_coefs_fs_left[vertices_in_region] = mean_region_coefs[region_name]
    
    for label_value in unique_labels_right:
        if label_value == 0:
            continue
        
        vertices_in_region = np.where(data_R == label_value)[0]
        region_name = f"region_{int(label_value)}_r"
        
        if region_name in mean_region_coefs:
            vertex_coefs_fs_right[vertices_in_region] = mean_region_coefs[region_name]
    
    # Map from FreeSurfer space back to original SurfeView space
    vertex_coefs_orig_left = np.zeros(10242)
    vertex_coefs_orig_right = np.zeros(10242)
    
    for fs_idx in range(10242):
        if fs_idx in fs_to_orig_left:
            orig_idx = fs_to_orig_left[fs_idx]
            vertex_coefs_orig_left[orig_idx] = vertex_coefs_fs_left[fs_idx]
    
    for fs_idx in range(10242):
        if fs_idx in fs_to_orig_right:
            orig_idx = fs_to_orig_right[fs_idx]
            vertex_coefs_orig_right[orig_idx] = vertex_coefs_fs_right[fs_idx]
    
    # Combine left and right
    vertex_coefs_orig = np.concatenate([vertex_coefs_orig_left, vertex_coefs_orig_right])
    
    print(f"Non-zero vertices: {np.sum(vertex_coefs_orig != 0)}")
    print(f"Min: {vertex_coefs_orig.min()}, Max: {vertex_coefs_orig.max()}")
    
    # Save coefficients
    output_file = output_dir / f"coefficients_mapping_{model_name}_boot{B}.npy"
    np.save(output_file, vertex_coefs_orig)
    print(f"Saved: {output_file}")
    
    # Also save as CSV for R
    df = pd.DataFrame({
        'vertex_id': np.arange(20484),
        'hemisphere': ['left']*10242 + ['right']*10242,
        'vertex_num': list(range(10242)) * 2,
        'mean_coef': vertex_coefs_orig
    })
    
    csv_file = output_dir / "vis_output" / f"mean_coef_mapping_{model_name}_boot{B}.csv"
    df.to_csv(csv_file, index=False)
    print(f"Saved: {csv_file}")

print("\n" + "="*60)
print("Coefficient extraction complete!")
print("="*60)

# GCN

In [ ]:
norm_y = True
icld_age_sex= False
scaler_name = "standard"
batch_size = 20
train_loader, val_loader, test_loader = input_to_graph(SurfeView_surfaces=SurfeView_surfaces, FVE_df_all=FVE_df_org,
                                                       partial_dat=partial_dat, scaler=scaler_name,
                                                       norm_y=norm_y, icld_age_sex=icld_age_sex, batch_size=batch_size)

In [ ]:
boolean_map = {0:'F', 1:'T'}
job_nickname = "GCN_std"
lr = 0.001
weight_decay = 0.0001
epochs = 200
patience = 5

job_full_nickname = f"{job_nickname}_partial{boolean_map[partial_dat]}_cov{boolean_map[icld_age_sex]}_p{patience}_norm{boolean_map[norm_y]}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(f"GCN_models/{job_full_nickname}.txt", "w") as f:
    print("Writing model output... \n", file=f, flush=True)
    print(f"Start training at {time.ctime(time.time())}", file=f, flush=True)
    print(f"epochs={epochs}, patience={patience}, batch_size={batch_size}, norm={norm_y}, partial={partial_dat}, age/sex={icld_age_sex}, scaler={scaler_name}, device={device}", file=f, flush=True)
    
    model = GCN(num_node_features=3).to(device)  


    trained_model = train_model(model=model, train_loader=train_loader, test_loader=val_loader,
                                lr=lr, weight_decay=weight_decay, epochs=epochs, patience=patience, log=f, device=device)
    

    
    torch.save(trained_model, f"GCN_models/{job_full_nickname}_full.pt")

    print(f"Start eval at {time.ctime(time.time())}", file=f, flush=True)


    trained_model.eval()
    y_pred, y_true = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)

            out = trained_model(batch.x, batch.edge_index, batch.batch)#, batch.edge_attr, batch.pos)

            y_pred.extend(out.cpu().numpy().flatten())  
            y_true.extend(batch.y.cpu().numpy().flatten())  

    y_pred = np.array(y_pred)
    y_true = np.array(y_true)

    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    print(f'R2 Score: {r2:.4f}, MSE: {mse:.4f}', file=f, flush=True)
    

 

In [ ]:
from FVE_GCN_script import *



# if retreiving from just weights
model = GCN(num_node_features=1).to(device)
state_dict = torch.load("GCN_models/best_models/base_adam_lr0.001_wd0.0001-org.pt")
model.load_state_dict(state_dict)
# if retreiving from just weights

# if retreiving from whole model
#model = torch.load("GCN_models/Base_adam_FVE_dat_p5_lr0.001_wd0.0001_full.pt")
#model.eval()
# if retreiving from whole model



y_pred, y_true = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)

        out = model(batch.x, batch.edge_index, batch.batch)#, batch.edge_attr, batch.pos)
        #out = model(batch.x, batch.edge_index, batch.batch)
        y_pred.extend(out.cpu().numpy().flatten())  
        y_true.extend(batch.y.cpu().numpy().flatten())  

y_pred = np.array(y_pred)
y_true = np.array(y_true)

SSRES = np.sum((y_true - y_pred)^2)
SSTOT = np.sum((y_true-np.mean(y_true))^2)
MyR2 = 1- (SSRES/SSTOT)         
r2 = r2_score(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
print(f'R2 Score: {r2:.4f}, myR2: {MyR2:.4f}, MSE: {mse:.4f}')

import matplotlib.pyplot as plt
plt.plot(y_pred, y_true)



# CNN

In [ ]:
def voxel_grid_to_tensor(voxel_grid, grid_shape=(139, 174, 128)):
    """
    Convert Open3D voxel grid with colors to a PyTorch 3D tensor.
    
    Args:
        voxel_grid (o3d.geometry.VoxelGrid): The Open3D voxel grid with colors.
        grid_shape (tuple): The shape of the output 3D tensor (default: (68, 171, 125)).
        
    Returns:
        torch.Tensor: 3D tensor of shape (68, 171, 125) with colors.
    """
    # Initialize a tensor of the given shape filled with zeros (for grayscale or single-channel data)
    tensor = torch.zeros(grid_shape)

    # Iterate over all the voxels in the voxel grid
    for voxel in voxel_grid.get_voxels():
        voxel_index = voxel.grid_index  # Voxel coordinates (i, j, k)
        voxel_color = voxel.color  # Corresponding voxel color (r, g, b)
        # print(voxel_color)
        
        # Convert the color to a single grayscale value, or you can store RGB values in 3 channels
        # gray_value = 0.2989 * voxel_color[0] + 0.5870 * voxel_color[1] + 0.1140 * voxel_color[2]
        
        # Ensure voxel index is within the grid shape bounds
        if all(0 <= voxel_index[i] < grid_shape[i] for i in range(3)):
            tensor[voxel_index[0], voxel_index[1], voxel_index[2]] = float(voxel_color[0])
    
    return tensor

In [ ]:
X_train_tensor = torch.load("data/X_train_tensor.pt")
X_test_tensor = torch.load("data/X_test_tensor.pt")
y_train_tensor = torch.load("data/y_train_tensor.pt")
y_test_tensor = torch.load("data/y_test_tensor.pt")

In [ ]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

X_tensor = None

X_new = X_train_tensor

points = vertices

for j in range(0,1000,1):
    texture = np.array(X_new[j,:])
    signals = texture
    # print(texture)

    colors = np.zeros((signals.shape[0], 3))  # Create a (n, 3) array filled with zeros
    colors[:, 0] = signals  # Set the first column to the original 1D array values
    # colors[:, 1] = 0.0
    # colors[:, 2] = 0.0


    # Create an Open3D point cloud object
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    # pcd = pcd.voxel_down_sample(voxel_size=10.0)

    # Voxelization
    voxel_size = 1.0  # Adjust voxel size based on your preference

    voxel_grid = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd,\
                                    voxel_size=voxel_size)

    # Get the voxel grid origin and the voxel size
    # origin = np.asarray(voxel_grid.origin)
    # voxel_size = voxel_grid.voxel_size
    # print(voxel_grid)

    # Create a dictionary to store signal values for each voxel
    voxel_signals = defaultdict(list)
    voxel_points = defaultdict(list)

    # Map each point to its corresponding voxel
#     for i, point in enumerate(points):
#         # Calculate voxel index (integer division of point position minus origin, divided by voxel size)
#         voxel_index = np.floor((point - origin) / voxel_size).astype(int)

#         # Store the signal in the corresponding voxel's list
#         voxel_signals[tuple(voxel_index)].append(signals[i])
#         voxel_points[tuple(voxel_index)] = point

    # Optionally, aggregate the signal (e.g., average) in each voxel
    # aggregated_voxel_signals = {voxel: np.mean(signal_list) for voxel, signal_list in voxel_signals.items()}


    # Visualization of the voxel grid and signal data
    # voxels = np.asarray([v.grid_index for v in voxel_grid.get_voxels()])
    # signals_to_plot = [aggregated_voxel_signals.get(tuple(v), 0) for v in voxels]
    
    input = voxel_grid_to_tensor(voxel_grid).unsqueeze(0).unsqueeze(0)
    X_tensor = input
    if j%100==0:
        print(f"{j}/{X_new.shape[0]}")
        
    X_tensor = X_tensor.squeeze(0)
    torch.save(X_tensor,f"CNN_data_out/X{j}.pt")
    shape = X_tensor.shape
    # print(X_tensor)
    del X_tensor
            # X_tensor = None
            # X_tensor = torch.load("X6.pt")
    
print(shape)
# print(max_x, max_y, max_z)



In [ ]:
X4_3dvox = torch.load("CNN_data_out/X4.pt")
print(X4_3dvox.shape)

plt.imshow(X4_3dvox[0,60,:,:], cmap=None)
plt.title("cut along axis=1")
plt.colorbar()
plt.show()

plt.imshow(X4_3dvox[0,:,130,:])
plt.title("cut along axis=2")
plt.colorbar()
plt.show()

plt.imshow(X4_3dvox[0,:,:,100])
plt.title("cut along axis=3")
plt.colorbar()
plt.show()

In [ ]:
!pip install natsort

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import glob
import os
from natsort import natsorted



class TensorDatasetFromFiles(Dataset):
    def __init__(self, data_dir, normalize=True, transform=None):
        self.X = torch.load(os.path.join(data_dir, 'X_train_tensor.pt'))
        self.X = natsorted(glob.glob(os.path.join(data_dir, 'X*.pt')))
        self.y = torch.load(os.path.join(data_dir, "y_train_tensor.pt"))

        self.transform = transform
        self.normalize = normalize


        if self.normalize:
            y_tensor = self.y
            y_tensor = (y_tensor - y_tensor.mean())/y_tensor.std()
            self.y = y_tensor
        
        assert len(self.X) == len(self.y), \
            "Number of input files must match the number of target samples"

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        sample_X = torch.load(self.X[idx])
        #sample_X = self.X[idx]#.squeeze(0)#.permute(2,0,1)
        sample_y = self.y[idx].unsqueeze(-1)#.squeeze(0)#.permute(2,0,1)

        if self.transform:
            sample_X = self.transform(sample_X)
        
        return sample_X, sample_y


# Paths to your data
data_dir="CNN_data_out"



# Create dataset and dataloader
dataset = TensorDatasetFromFiles(data_dir, transform=None)